### List of Libraries/Dependencies:
Requires ~40s to import all libraries.

In [1]:
# Document Converter
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import re
import unicodedata
import json
import logging
import time
from datetime import datetime
from collections.abc import Iterable
from collections import defaultdict
from pathlib import Path

from docling_core.types.doc.base import ImageRefMode
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.document import ConversionResult
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption, HTMLFormatOption

# Schema Loader
# Cite the libraryyyyyyyyyyyyyyyyyyyyyyyyyyyyyy
from rdflib import RDF, RDFS, OWL, URIRef, Namespace, Literal, Dataset, Graph
from rdflib.namespace import XSD
import rdflib
import os

# Hybrid Chunker
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (DOCLING)
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY (HUGGINGFACE)
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from transformers import AutoTokenizer
from docling.document_converter import DocumentConverter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any

# Chunk Processor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import spacy
from spacy.symbols import VERB, AUX

# Triple Extractor
# CITE THE LIBRARYYYYYYYYYYYYYYYYYY
import openai
import os

# Triple Extractor - Batch Processing
from concurrent.futures import ThreadPoolExecutor
import json
import time

# Triple Sanitizer
import json

# E-R Normalizer
# 1. MinHash LSH
from datasketch import MinHash, MinHashLSH

# E-R Normalizer
# 2. FAISS Indexing
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Graph Generator
import copy

# Graph Visualizer
from pyvis.network import Network
import streamlit.components.v1 as components

2026-07-09 00:44:47,786 - INFO - Loading faiss with AVX2 support.
2026-07-09 00:44:47,926 - INFO - Successfully loaded faiss with AVX2 support.


### Document Converter

In [2]:
# Cleans documents
def clean_text(text: str) -> str:
    if not text:
        return text

    # 1. Normalize Unicode (fixes odd composed characters)
    text = unicodedata.normalize("NFKC", text)

    # 2. Replace the following:
    replacements = {
        "\u00A0": " ",  # NBSP
        "\u202F": " ",  # narrow NBSP
        "\u202f": " ",  # narrow NBSP
        "\u2009": " ",  # thin space
        "\u2007": " ",  # figure space
        "\x00": "",     # null bytes
        "\ufffd": ""    # Unicode replacement char
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    # 4. Remove control characters (but keep newlines/tabs)
    text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", text)

    return text

_log = logging.getLogger(__name__)

# Export toggles:
# - USE_V2 controls modern Docling document exports.
# - USE_LEGACY enables legacy Deep Search exports for comparison or migration.
USE_V2 = True
USE_LEGACY = False


def export_documents(
    conv_results: Iterable[ConversionResult],
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    failure_count = 0
    partial_success_count = 0

    for conv_res in conv_results:
        if conv_res.status == ConversionStatus.SUCCESS:
            success_count += 1
            doc_filename = conv_res.input.file.stem

            if USE_V2:
                # Export converted files as markdown files
                conv_res.document.save_as_markdown(
                    output_dir / f"{doc_filename}.md",
                    image_mode=ImageRefMode.PLACEHOLDER,
                )
                
                # conv_res.document.save_as_json(
                #     output_dir / f"{doc_filename}.json",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                # )
                
                # conv_res.document.save_as_markdown(
                #     output_dir / f"{doc_filename}.txt",
                #     image_mode=ImageRefMode.PLACEHOLDER,
                #     strict_text=True,
                # )

                # Export Docling document format to markdown:
                with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Docling document format to text:
                # with (output_dir / f"{doc_filename}.txt").open("w") as fp:
                #     fp.write(conv_res.document.export_to_markdown(strict_text=True))

            if USE_LEGACY:
                # Export Markdown format:
                with (output_dir / f"{doc_filename}.legacy.md").open("w", encoding="utf-8") as fp:
                    raw_md = conv_res.document.export_to_markdown()
                    fp.write(clean_text(raw_md))

                # # Export Deep Search document JSON format:
                # with (output_dir / f"{doc_filename}.legacy.json").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(json.dumps(conv_res.document.export_to_dict()))

                # # Export Text format:
                # with (output_dir / f"{doc_filename}.legacy.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(
                #         conv_res.document.export_to_markdown(strict_text=True)
                #     )

                # # Export Document Tags format:
                # with (output_dir / f"{doc_filename}.legacy.doctags.txt").open(
                #     "w", encoding="utf-8"
                # ) as fp:
                #     fp.write(conv_res.document.export_to_doctags())

        elif conv_res.status == ConversionStatus.PARTIAL_SUCCESS:
            _log.info(
                f"Document {conv_res.input.file} was partially converted with the following errors:"
            )
            for item in conv_res.errors:
                _log.info(f"\t{item.error_message}")
            partial_success_count += 1
        else:
            _log.info(f"Document {conv_res.input.file} failed to convert.")
            failure_count += 1

    _log.info(
        f"Processed {success_count + partial_success_count + failure_count} docs, "
        f"of which {failure_count} failed "
        f"and {partial_success_count} were partially converted."
    )
    return success_count, partial_success_count, failure_count


def main():
    logging.basicConfig(level=logging.INFO)

    # Location of source documents
    data_folder = Path("./eu_legislation")
    input_doc_paths = [file_path for file_path in data_folder.iterdir()]

    # buf = BytesIO((data_folder / "pdf/2206.01062.pdf").open("rb").read())
    # docs = [DocumentStream(name="my_doc.pdf", stream=buf)]
    # input = DocumentConversionInput.from_streams(docs)

    # # Turn on inline debug visualizations:
    # settings.debug.visualize_layout = True
    # settings.debug.visualize_ocr = True
    # settings.debug.visualize_tables = True
    # settings.debug.visualize_cells = True

    # Configure the PDF pipeline. Enabling page image generation improves HTML
    # previews (embedded images) but adds processing time.
    pdf_pipeline_options = PdfPipelineOptions()
    pdf_pipeline_options.generate_page_images = True

    doc_converter = DocumentConverter(
        allowed_formats=[
            InputFormat.PDF,
            InputFormat.HTML
        ],
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pdf_pipeline_options,
                backend=DoclingParseV4DocumentBackend
            ),
            InputFormat.HTML: HTMLFormatOption()
        }
    )

    start_time = time.time()

    # Convert all inputs. Set `raises_on_error=False` to keep processing other
    # files even if one fails; errors are summarized after the run.
    conv_results = doc_converter.convert_all(
        input_doc_paths,
        raises_on_error=False,  # to let conversion run through all and examine results at the end
    )
    # Write outputs to ./scratch and log a summary.
    _success_count, _partial_success_count, failure_count = export_documents(
        conv_results, output_dir=Path("converted_docs")
    )

    end_time = time.time() - start_time

    _log.info(f"Document conversion complete in {end_time:.2f} seconds.")

    if failure_count > 0:
        raise RuntimeError(
            f"The example failed converting {failure_count} on {len(input_doc_paths)}."
        )

if __name__ == "__main__":
    main()

2026-07-09 00:44:49,643 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-07-09 00:44:49,660 - INFO - Going to convert document batch...
2026-07-09 00:44:49,664 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-07-09 00:44:49,694 - INFO - Loading plugin 'docling_defaults'
2026-07-09 00:44:49,701 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-07-09 00:44:49,702 - INFO - Processing document 31953D0030en.html
2026-07-09 00:44:49,713 - INFO - Finished converting document 31953D0030en.html in 0.06 sec.
2026-07-09 00:44:49,726 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-07-09 00:44:49,737 - INFO - Going to convert document batch...
2026-07-09 00:44:49,738 - INFO - Processing document 31954S0024en.html
2026-07-09 00:44:49,742 - INFO - Finished converting document 31954S0024en.html in 0.03 sec.
2026-07-09 00:44:49,752 - INFO - detected formats: [<InputFormat.HTML: 'html'>]
2026-07-09 00:44:49,761

### Schema Loader

In [3]:
schema_folder = "./schema"
schema_files = [os.path.join(schema_folder, file) for file in os.listdir(schema_folder)]

schema_files

['./schema\\cdm.rdf', './schema\\cdm_datatypes.rdf']

Add predefined or known namespaces, if any:

In [4]:
def extract_resources(graph, resource_type):
    resources = {}

    resource_types = {
        "class": OWL.Class,
        "obj_prop": OWL.ObjectProperty,
        "data_prop": OWL.DatatypeProperty,
        "datatype": RDFS.Datatype
    }

    chosen_type = resource_types[resource_type]

    for s in set(graph.subjects(RDF.type, chosen_type)):
        if not isinstance(s, rdflib.term.BNode):
            label = graph.value(s, RDFS.label)

            if label:
                resources[str(s)] = str(label)
            else:
                # fallback to URI fragment
                resources[str(s)] = s.split("#")[-1]

    return resources

def resource_text(graph, uri):
    label = graph.value(uri, RDFS.label)
    comment = graph.value(uri, RDFS.comment)

    return " ".join(
        x for x in [
            str(label) if label else "",
            str(comment) if comment else ""
        ]
        if x
    )

In [5]:
schema_graph = Graph()

index = 1
for file in schema_files:
    schema_graph.parse(file, format="xml")
    print(f"Finished parsing RDF file {index}.")
    index += 1

print()

# Only retrieves declared namespaces
schema_namespaces = set(schema_graph.namespaces())
# schema_namespaces.remove(('', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))
# schema_namespaces.add(('cdmplus#', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdmplus#')))

print("Schema Namespaces:")
for ns in schema_namespaces:
    print(ns)

print()

# Extract classes
classes = extract_resources(schema_graph, "class")
print(f"Found {len(classes)} classes.")

# Extract object properties
obj_properties = extract_resources(schema_graph, "obj_prop")
print(f"Found {len(obj_properties)} object properties.")

# Extract datatype properties
datatype_properties = extract_resources(schema_graph, "data_prop")
print(f"Found {len(datatype_properties)} datatype properties.")

# Extract datatypes
datatypes = extract_resources(schema_graph, "datatype")
print(f"Found {len(datatypes)} datatypes.")

# Extract domain and range for all properties
prop_domains = dict()
prop_ranges = dict()

for prop in set(obj_properties.keys()).union(set(datatype_properties.keys())):
    domains = set(schema_graph.objects(prop, RDFS.domain))
    ranges = set(schema_graph.objects(prop, RDFS.range))
    prop_domains[prop] = domains
    prop_ranges[prop] = ranges

# Pretty-print helper function
def pretty_uri(uri):
    if isinstance(uri, URIRef):
        return str(uri).split('#')[-1]
    return str(uri)

print("\nSample property info:")
for prop in list(prop_domains.keys())[:5]:
    print(f"{pretty_uri(prop)}:")
    print(f"\tDomains:\t{[pretty_uri(d) for d in prop_domains[prop]]}")
    print(f"\tRanges:\t{[pretty_uri(r) for r in prop_ranges[prop]]}")


Finished parsing RDF file 1.
Finished parsing RDF file 2.

Schema Namespaces:
('admin', rdflib.term.URIRef('http://publications.europa.eu/ontology/cdm/admin#'))
('brick', rdflib.term.URIRef('https://brickschema.org/schema/Brick#'))
('void', rdflib.term.URIRef('http://rdfs.org/ns/void#'))
('geo', rdflib.term.URIRef('http://www.opengis.net/ont/geosparql#'))
('sh', rdflib.term.URIRef('http://www.w3.org/ns/shacl#'))
('prov', rdflib.term.URIRef('http://www.w3.org/ns/prov#'))
('csvw', rdflib.term.URIRef('http://www.w3.org/ns/csvw#'))
('qb', rdflib.term.URIRef('http://purl.org/linked-data/cube#'))
('xsd', rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#'))
('doap', rdflib.term.URIRef('http://usefulinc.com/ns/doap#'))
('dcterms', rdflib.term.URIRef('http://purl.org/dc/terms/'))
('rdf', rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#'))
('wgs', rdflib.term.URIRef('https://www.w3.org/2003/01/geo/wgs84_pos#'))
('annotation', rdflib.term.URIRef('http://publications.europa.eu/o

#### Resource-URI dictionaries

In [6]:
rdf_resources = list(classes.keys()) + list(obj_properties.keys()) + list(datatype_properties.keys()) + list(datatypes.keys())
rdf_dict = {**classes, **obj_properties, **datatype_properties, **datatypes}

##### *Inspecting the Retrieved Classes:*

In [7]:
for resource in classes:
    text = resource_text(schema_graph, resource)
    if text.strip() != "":
        print(text)

In [8]:
list(classes.items())

[('http://publications.europa.eu/ontology/cdm#ATTO_FD_577',
  'FD_577 ATTO table'),
 ('http://publications.europa.eu/ontology/cdm#concept_currency',
  'Concept representing a currency'),
 ('http://publications.europa.eu/ontology/cdm#report-synthesis',
  'Report synthesis'),
 ('http://publications.europa.eu/ontology/cdm#ATTO_FD_380',
  'FD_380 ATTO table'),
 ('http://publications.europa.eu/ontology/cdm#ATTO_FD_557',
  'FD_557 ATTO table'),
 ('http://publications.europa.eu/ontology/cdm#proceedings', 'Proceedings'),
 ('http://publications.europa.eu/ontology/cdm#report-project',
  'Report project'),
 ('http://publications.europa.eu/ontology/cdm#com_final',
  'Digitised COM final document'),
 ('http://publications.europa.eu/ontology/cdm#fragment-budget-general-draft',
  'Fragment of draft general budget'),
 ('http://publications.europa.eu/ontology/cdm#regulation_delegated',
  'Delegated regulation'),
 ('http://publications.europa.eu/ontology/cdm#special-official-journal',
  'Special Edition

In [9]:
print("No. of RDF Resources:", len(rdf_resources))

No. of RDF Resources: 2907


### Hybrid Chunker

Prepare a Custom Data Structure

In [10]:
@dataclass
class DocumentState:
    raw_chunks: list = field(default_factory=list)
    processed_chunks: list = field(default_factory=list)
    extracted_triples: dict = field(default_factory=dict)
    candidate_cache: dict = field(default_factory=dict)
    resolved_cache: dict = field(default_factory=dict)

In [11]:
EMBED_MODEL_ID = "openai/gpt-oss-120b"

MAX_TOKENS = 1024

tokenizer = HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(EMBED_MODEL_ID),
    max_tokens=MAX_TOKENS,
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,
)

digitalizer = DocumentConverter()

conv_docs_folder = Path("./converted_docs")
conv_doc_paths = [file_path for file_path in conv_docs_folder.iterdir()]

doc_states = dict()

for doc_path in conv_doc_paths:
    filename = os.path.basename(doc_path)

    doc = digitalizer.convert(source=doc_path).document
    raw_chunks = list(chunker.chunk(dl_doc=doc))

    doc_states[filename] = DocumentState(
        raw_chunks=[(f"chunk_{i + 1}", chunk) for i, chunk in enumerate(raw_chunks)]
        )

2026-07-09 00:44:56,822 - INFO - detected formats: [<InputFormat.MD: 'md'>]
2026-07-09 00:44:56,824 - INFO - Going to convert document batch...
2026-07-09 00:44:56,826 - INFO - Initializing pipeline for SimplePipeline with options hash 995a146ad601044538e6a923bea22f4e
2026-07-09 00:44:56,827 - INFO - Processing document 31953D0030en.md
2026-07-09 00:44:56,866 - INFO - Finished converting document 31953D0030en.md in 0.05 sec.
2026-07-09 00:44:57,119 - INFO - detected formats: [<InputFormat.MD: 'md'>]
2026-07-09 00:44:57,121 - INFO - Going to convert document batch...
2026-07-09 00:44:57,123 - INFO - Processing document 31954S0024en.md
2026-07-09 00:44:57,152 - INFO - Finished converting document 31954S0024en.md in 0.03 sec.
2026-07-09 00:44:57,167 - INFO - detected formats: [<InputFormat.MD: 'md'>]
2026-07-09 00:44:57,169 - INFO - Going to convert document batch...
2026-07-09 00:44:57,170 - INFO - Processing document 31954S0026en.md
2026-07-09 00:44:57,201 - INFO - Finished converting d

##### *Chunk Inspection:*

In [12]:
sample_file = list(doc_states.keys())[0]
sample_chunk_list = doc_states[sample_file].raw_chunks

print("The current chunks are from:", sample_file, "\n")
for i, chunk in sample_chunk_list:
    print(f"=== {i} ===")
    txt_tokens = tokenizer.count_tokens(chunk.text)
    print(f"chunk.text ({txt_tokens} tokens):\n{chunk.text!r}")

    ser_txt = chunker.contextualize(chunk=chunk)
    ser_tokens = tokenizer.count_tokens(ser_txt)
    print(f"chunker.contextualize(chunk) ({ser_tokens} tokens):\n{ser_txt!r}")

    print()    

The current chunks are from: 31953D0030en.md 

=== chunk_1 ===
chunk.text (303 tokens):
"**ECSC High Authority: Decision No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel**\n*Official Journal 006 , 04/05/1953 P. 0109 - 0110 Danish special edition: Series I Chapter 1952-1958 P. 0009 English special edition: Series I Chapter 1952-1958 P. 0009 Greek special edition: Chapter 08 Volume 1 P. 0005 Spanish special edition: Chapter 08 Volume 1 P. 0005 Portuguese special edition Chapter 08 Volume 1 P. 0005 Finnish special edition: Chapter 12 Volume 3 P. 0003 Swedish special edition: Chapter 12 Volume 3 P. 0003*\nDECISION No 30-53  of 2 May 1953  on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of

### Chunk Processor

In [13]:
nlp = spacy.load("en_core_web_lg")

def strip_all_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# ---------------------------------------------------------------------
# Flatten all chunks into a processing queue
# ---------------------------------------------------------------------

jobs = list()

for filename, doc_state in doc_states.items():

    for chunk in doc_state.raw_chunks:
        jobs.append((filename, chunk))

docs = nlp.pipe(
    (chunk[1].text for _, chunk in jobs),
    batch_size=64,
    n_process=2
)

# ---------------------------------------------------------------------
# Process chunks
# ---------------------------------------------------------------------

processed_chunks = list()

for (filename, (chunk_id, _)), chunk in zip(jobs, docs):

    filtered_sents = list()

    for sent in chunk.sents:
        text = sent.text.strip()

        if not text:
            continue

        # check for VERB or AUX in the sentence
        has_verb_or_aux = any(
            token.pos_ in {"VERB", "AUX"} for token in sent
        )

        if not has_verb_or_aux:
            continue

        filtered_sents.append(text)

    if not filtered_sents:
        continue

    processed_chunk = " ".join(filtered_sents)

    cleaned = strip_all_whitespace(processed_chunk)

    if cleaned:  # simpler + safer than comparing twice
        processed_chunks.append({
            "filename": filename,
            "chunk_id": chunk_id,
            "content": cleaned
        })
        doc_states[filename].processed_chunks.append((chunk_id, cleaned))

    print("CHUNK:", repr(cleaned))
    print("FILE:", filename, "\n")

CHUNK: "No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel** *Official Journal 006 , 04/05/1953 P. 0109 - 0110 Danish special edition: Series I Chapter 1952-1958 P. 0009 Finnish special edition: Chapter 12 Volume 3 P. 0003 Swedish special edition: Chapter 12 Volume 3 P. 0003* DECISION No 30-53 of 2 May 1953 on practices prohibited by Article 60 (1) of the Treaty in the common market for coal and steel THE HIGH AUTHORITY, Having regard to Article 60 and Article 63 (2) of the Treaty; Whereas compliance with the obligations of non-discrimination involves uniform application by undertakings of the conditions shown in their price lists with no other increases or reductions and no evasion of those obligations by allowing longer periods for settlement without a corresponding increase in price; Whereas the exception to this rule, namely the option of aligning a quotation on a competitor's price list,"
FILE: 31953D0030en.md 

C

### Triple Extractor

In [208]:
# Security Measure
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')

client = openai.OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key = openai.api_key
    )

extraction_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {

            "subject": {
                "type": "string",
                "minLength": 1
            },

            "predicate": {
                "type": "string",
                "minLength": 1
            },

            "object": {
                "type": "string",
                "minLength": 1
            },
        },

        "required": [
            "subject",
            "predicate",
            "object"
        ],

        "additionalProperties": False
    }
}

extraction_prompt = """
You will be given a chunk of text from an EU legal document.

Extract the legal semantic information as a collection of triples in JSON.

### Extraction rules

1. Extract only information that is explicitly stated or clearly implied by the text. Do not invent facts or relationships.

2. Ignore document formatting, publication metadata, pagination, headers, footers, chapter numbers, volume numbers, page numbers, and other editorial or bibliographic information unless they are explicitly referenced by the legal provisions.

   Do not extract:

   * (Swedish Special Edition, chapter, 12)
   * (Swedish Special Edition, volume, 3)
   * (Swedish Special Edition, page, 0003)

3. Resolve pronouns, demonstratives, and other references to the entities they refer to.

4. Preserve entity names in the subject and object components as faithful noun phrases. Keep only modifiers that are necessary to preserve legal meaning (e.g. jurisdiction, role, legal status). Do not include full clauses or conditional/procedural phrases.

   Good:
   * (producer established in a Member State, submits, application)
   * (Article 5, lays down, conditions for market access)

   Too minimal:
   * (producer, submits, application)

   Too long:
   * (producer established in a Member State in accordance with Article 5 and operating under transitional provisions, submits, application)
   * (application, subject to, all conditions laid down in Articles 6–8 unless otherwise specified)

5. Express predicates as concise relation phrases while preserving their meaning. Do not incorporate conditions, exceptions, or subordinate clauses into the predicate.

   Instead of:
   * (application, is subject to the conditions laid down in, Article 5)

   Extract:
   * (application, subject to, Article 5)

6. If a sentence contains a condition, exception, or subordinate clause (containing "if", "unless", "provided that", or similar expressions), extract the principal relation.

7. If an entity is explicitly stated, or its class is unambiguous from the surrounding context, represent the relationship using the predicate "type of".

   Examples:
   * (Regulation (EU) 2024/1234, type of, Regulation)
   * (Article 5, type of, Article)
   * (European Commission, type of, Institution)
"""

def get_completion(system_prompt="", query="", schema=extraction_schema, with_reasoning=True):
    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
           {'role':'system', 'content': system_prompt},
           {'role':'user', 'content': query}
           ],
        temperature=0,
        top_p=0.1, # Test different values
        max_tokens=None,
        stream=False,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "triples",
                "schema": schema
                }
            }
        )
    
    reasoning = ""
    if with_reasoning:
        reasoning = getattr(completion.choices[0].message, "reasoning_content", None)

    return completion.choices[0].message.content


#### Batch Processing

In [211]:
# Parse outputs
responses = list()

def extraction_thread(chunk_dict):
    filename = chunk_dict["filename"]
    chunk_id = chunk_dict["chunk_id"]
    
    response = get_completion(
        system_prompt=extraction_prompt,
        query=chunk_dict["content"],
        schema=extraction_schema
    )

    result = json.loads(response)

    return {"filename": filename, "chunk_id": chunk_id, "triples": result}

with ThreadPoolExecutor(max_workers=15) as executor:
    responses = list(executor.map(extraction_thread, processed_chunks))

for element in responses:
    filename, chunk_id, result = element.values()
    doc_states[filename].extracted_triples[chunk_id] = list()
    triple_list = doc_states[filename].extracted_triples[chunk_id]

    if type(result) == str:
        triple_list.append(result)
    elif type(result) == list:
        triple_list.extend(result)

2026-07-09 05:07:50,073 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-07-09 05:07:50,078 - INFO - Retrying request to /chat/completions in 0.383204 seconds
2026-07-09 05:07:50,100 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-07-09 05:07:50,118 - INFO - Retrying request to /chat/completions in 0.434854 seconds
2026-07-09 05:07:50,306 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-07-09 05:07:50,310 - INFO - Retrying request to /chat/completions in 0.468049 seconds
2026-07-09 05:07:53,039 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 05:07:53,200 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 05:07:57,842 - INFO - HTTP Request: POST https://integra

In [16]:
raw_triples = [
    ((filename, chunk_id), triple)
    for filename, doc in list(doc_states.items())[:]
    for chunk_id, triple_list in doc.extracted_triples.items()
    for triple in triple_list
]

In [51]:
nlp = spacy.load("en_core_web_sm")

# ---------------------------------------------------------------------
# Normalization
# ---------------------------------------------------------------------

def normalize_entity(text: str) -> str:
    """Deterministic cleanup of entity names."""
    text = re.sub(r"\s+", " ", text.strip())
    text = re.sub(r"^(the|The)\s+", "", text)
    text = text.rstrip(".,;:")
    return text


def normalize_predicate(text: str) -> str:
    """Deterministic cleanup of predicates."""
    text = re.sub(r"\s+", " ", text.strip())

    replacements = {
        "is obliged to": "obligated to",
        "is required to": "required to",
        "is subject to": "subject to",
        "is entitled to": "entitled to",
    }

    lower = text.lower()

    for old, new in replacements.items():
        if lower.startswith(old):
            text = new + text[len(old):]
            break

    return text


# ---------------------------------------------------------------------
# Composite entity detection
# ---------------------------------------------------------------------

def split_entity(text: str):
    """
    Returns:
        None              -> entity should not be expanded
        list[str]         -> safe deterministic expansion
    """

    doc = nlp(text)

    # Require at least one coordination
    if not any(tok.dep_ == "conj" for tok in doc):
        return None

    # Reject nested structures
    if any(tok.dep_ in {"relcl", "advcl", "ccomp", "xcomp"} for tok in doc):
        return None

    noun_chunks = list(doc.noun_chunks)

    # Conservative:
    # Only expand if every noun chunk is a simple coordinated NP.
    if len(noun_chunks) < 2:
        return None

    entities = [chunk.text.strip() for chunk in noun_chunks]

    # Remove duplicates while preserving order
    entities = list(dict.fromkeys(entities))

    return entities if len(entities) > 1 else None


# ---------------------------------------------------------------------
# Triple expansion
# ---------------------------------------------------------------------

def expand_triple(triple: dict):
    """
    Returns a list of normalized triples.
    """

    subject = normalize_entity(triple["subject"])
    predicate = normalize_predicate(triple["predicate"])
    obj = normalize_entity(triple["object"])

    subjects = split_entity(subject) or [subject]
    objects = split_entity(obj) or [obj]

    expanded = []

    for s in subjects:
        for o in objects:

            expanded.append({
                "subject": s,
                "predicate": predicate,
                "object": o,
            })

    return expanded


# ---------------------------------------------------------------------
# Batch normalization
# ---------------------------------------------------------------------

def normalize_triples(triples_with_ids):
    """
    Normalize and expand an extracted triple list.
    """

    normalized = []

    for id, triple in triples_with_ids:
        expanded_list = [(id, tr) for tr in expand_triple(triple)]
        normalized.extend(expanded_list)

    return normalized

In [157]:
normalized_triples = normalize_triples(raw_triples)
normalized_triples = [tr for tr in normalized_triples if type(tr[0]) == tuple]

### Semantic Filter

In [112]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

resources = {
    "class": classes,
    "obj_prop": obj_properties,
    "datatype_prop": datatype_properties,
    "datatype": datatypes,
}

index_lookup = {}

for resource_type, resource_dict in resources.items():
    labels = list(resource_dict.values())
    uris = list(resource_dict.keys())

    embeddings = emb_model.encode(
        labels,
        normalize_embeddings=True
    )

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    index_lookup[resource_type] = (index, uris, labels)

2026-07-09 02:50:06,512 - INFO - Use pytorch device_name: cpu
2026-07-09 02:50:06,513 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [159]:
TYPE_MAP = {
    "string": XSD.string,
    "integer": XSD.integer,
    "decimal": XSD.decimal,
    "boolean": XSD.boolean,
    "date": XSD.date,
    "datetime": XSD.dateTime,
}

def detect_literal_type(value: str):

    v = value.strip()

    # boolean
    if v.lower() in {"true", "false"}:
        return "boolean"

    # integer
    if re.fullmatch(r"-?\d+", v):
        return "integer"

    # decimal
    if re.fullmatch(r"-?\d+\.\d+", v):
        return "decimal"

    # date (YYYY-MM-DD)
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", v):
        return "date"

    # datetime (very permissive ISO-ish)
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}", v):
        return "datetime"

    return "class"

# --------------------------------------------------
# 1. Collect lookup requests
# --------------------------------------------------

lookup_requests = defaultdict(list)

for idx, (_, triple) in enumerate(normalized_triples):

    # Subject -> always class lookup for now
    lookup_requests["class"].append({
        "idx": idx,
        "field": "subject",
        "text": triple["subject"]
    })

    # Object
    object_type = detect_literal_type(triple["object"])

    if object_type == "class":
        lookup_requests["class"].append({
            "idx": idx,
            "field": "object",
            "text": triple["object"]
        })

    else:
        # Handle literals immediately
        if object_type == "integer":
            value = int(triple["object"])
        elif object_type == "decimal":
            value = float(triple["object"])
        else:
            value = triple["object"]

        triple["object"] = Literal(
            value,
            datatype=TYPE_MAP[object_type]
        )


    # Predicate
    if triple["predicate"] == "type of":

        # Directly assign RDF predicate
        triple["predicate"] = RDF.type
        triple["predicate_cands"] = None

    else:

        prop_type = (
            "obj_prop"
            if object_type == "class"
            else "datatype_prop"
        )

        lookup_requests[prop_type].append({
            "idx": idx,
            "field": "predicate",
            "text": triple["predicate"]
        })


# --------------------------------------------------
# 2. Batch semantic lookup
# --------------------------------------------------

lookup_results = {}


for resource_type, requests in lookup_requests.items():

    texts = [
        r["text"]
        for r in requests
    ]

    embeddings = emb_model.encode(
        texts,
        batch_size=128,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    index, uris, labels = index_lookup[resource_type]

    scores, indices = index.search(
        embeddings,
        3
    )

    for req, score_row, idx_row in zip(
        requests,
        scores,
        indices
    ):

        candidates = [
            (
                labels[i],
                uris[i],
                float(score_row[j])
            )
            for j, i in enumerate(idx_row)
        ]

        lookup_results[
            (req["idx"], req["field"], req["text"])
        ] = candidates


# --------------------------------------------------
# 3. Write results into a list
# --------------------------------------------------

scored_triples = copy.deepcopy(normalized_triples)

for (idx, field, _), candidates in lookup_results.items():

    triple = scored_triples[idx][1]

    best = candidates[0]
    score = best[2]

    if score > 0.7:
        triple[field] = URIRef(best[1])
        triple[f"{field}_cands"] = None

    elif score > 0.6:
        triple[f"{field}_cands"] = candidates

    else:
        triple[f"{field}_cands"] = None

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Batches:   0%|          | 0/74 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [160]:
subject_count = ["x" for _, tr in scored_triples if tr.get("subject_cands")]
predicate_count = ["x" for _, tr in scored_triples if tr.get("predicate_cands")]
object_count = ["x" for _, tr in scored_triples if tr.get("object_cands")]

print(len(subject_count), len(predicate_count), len(object_count))

len(subject_count + predicate_count + object_count)

1294 1313 920


3527

In [178]:
def collect_llm_review_queue(scored_triples):
    """
    Collect components already flagged as ambiguous by semantic matching.
    """

    queue = dict()

    for idx, (location, triple) in enumerate(scored_triples):

        for component in ["subject", "predicate", "object"]:

            candidates = triple.get(f"{component}_cands")

            if candidates is None:
                continue

            queue.setdefault(location, []).append({
                "triple_idx": idx,
                "component": component,
                "value": triple[component],
                "candidates": [
                    {
                        "label": label,
                        "uri": str(uri),
                        "score": score
                    }
                    for label, uri, score in candidates
                ]
            })

    return queue

In [179]:
review_queue = collect_llm_review_queue(scored_triples)

In [231]:
def format_review_chunk(location, review_items, scored_triples):
    """
    Format one chunk for LLM review.

    Parameters
    ----------
    location : (filename, chunk_id)
    review_items : list of ambiguous components
    normalized_triples : list of
        ((filename, chunk_id), triple)
    """
    
    filename, chunk_id = location

    lines = [
        f"Document: {filename}",
        f"Chunk: {chunk_id}",
        ""
    ]

    # Group ambiguous components by triple
    grouped = dict()

    for item in review_items:
        grouped.setdefault(item["triple_idx"], []).append(item)

    # Print each affected triple once
    for triple_idx in sorted(grouped):

        _, triple = scored_triples[triple_idx]

        lines.append(f"Triple {triple_idx}")
        lines.append(f"Subject: {triple['subject']}")
        lines.append(f"Predicate: {triple['predicate']}")
        lines.append(f"Object: {triple['object']}")
        lines.append("")

        for item in grouped[triple_idx]:
            lines.append(f"Review {item['component']}")

            for rank, cand in enumerate(item["candidates"], start=1):
                label = cand["label"]
                desc = f"{rank}. {label}"

                lines.append(desc)

            lines.append("")

        lines.append("-" * 40)

    return "\n".join(lines)

def build_review_queries(review_queue, scored_triples):

    queries = list()

    for location, review_items in review_queue.items():

        queries.append({
            "location": location,
            "query": format_review_chunk(
                location,
                review_items,
                scored_triples
            )
        })

    return queries

In [232]:
review_queries = build_review_queries(review_queue, scored_triples)

In [233]:
print(review_queries[14]["query"])

Document: 31958R0003_01_en.md
Chunk: chunk_3

Triple 487
Subject: http://publications.europa.eu/ontology/cdm#communication_ec
Predicate: classifies provisionally
Object: information covered by Article 24 of the Treaty

Review object
1. Treaty
2. Concept representing a treaty
3. Consolidated treaty

----------------------------------------
Triple 488
Subject: http://publications.europa.eu/ontology/cdm#communication_ec
Predicate: classifies definitively
Object: information covered by Article 24 of the Treaty

Review object
1. Treaty
2. Concept representing a treaty
3. Consolidated treaty

----------------------------------------
Triple 492
Subject: http://publications.europa.eu/ontology/cdm#communication_ec
Predicate: notifies
Object: competent institutions

Review object
1. Institutional arrangement
2. Concept representing an institution
3. Local authority

----------------------------------------
Triple 494
Subject: http://publications.europa.eu/ontology/cdm#communication_ec
Predicate:

In [202]:
disambiguation_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "triple_id": {
                "type": "integer",
                "minimum": 0
            },
            "component": {
                "type": "string",
                "enum": [
                    "subject",
                    "predicate",
                    "object"
                ]
            },
            "selection": {
                "type": [
                    "integer",
                    "null"
                ],
                "minimum": 1,
                "description": "1-based index of the selected candidate. Use null if none are suitable."
            }
        },
        "required": [
            "triple_id",
            "component",
            "selection"
        ],
        "additionalProperties": False
    }
}

disambiguation_prompt = """
You are given extracted legal triples and candidate RDF resources.

Select the most appropriate candidate for each ambiguous component.

Rules:
- Choose a candidate only if it matches the meaning of the component.
- Do not choose a candidate only because the wording is similar.
- If no candidate is suitable, return null.

Input:

"""

In [270]:
# Parse outputs
nested_list = list()
disambiguated_components = list()


def disambiguation_thread(context_dict):
    query = context_dict["query"]
    response = get_completion(
        system_prompt=disambiguation_prompt,
        query=query,
        schema=disambiguation_schema,
        with_reasoning=False
    )

    result = json.loads(response)

    return result

with ThreadPoolExecutor(max_workers=10) as executor:
    nested_list = list(executor.map(disambiguation_thread, review_queries))

disambiguated_components = [
    element
    for sublist in nested_list
    for element in sublist
]

2026-07-09 13:22:48,850 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:22:49,100 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:22:50,737 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:22:52,044 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:22:54,626 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:22:56,334 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:22:57,779 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:23:01,303 - INFO - HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-07-09 13:23

In [281]:
FIELD_MAP = {
    "subject": ("subject", "subj_cands"),
    "predicate": ("predicate", "pred_cands"),
    "object": ("object", "obj_cands"),
}


def apply_disambiguation(scored_triples, disambiguated_components):
    """
    Replace ambiguous components with the URI selected by the LLM.
    """

    for decision in disambiguated_components:

        triple_idx = decision["triple_id"]
        component = decision["component"]
        selection = decision["selection"]

        if selection is None:
            continue

        _, triple = scored_triples[triple_idx]

        value_field, cand_field = FIELD_MAP[component]

        candidates = triple.get(cand_field)

        if not candidates:
            continue

        # Convert 1-based index -> Python index
        label, uri, score = candidates[selection - 1]

        triple[value_field] = uri
        triple[cand_field] = None

def collect_unresolved_components(disambiguated_triples):
    """
    Collect every component that still requires a custom RDF resource.
    """

    unresolved = []

    field_map = [
        ("subject", "subj_cands"),
        ("predicate", "pred_cands"),
        ("object", "obj_cands"),
    ]

    for triple_idx, (_, triple) in enumerate(disambiguated_triples):

        for value_field, cand_field in field_map:
            if type(triple[value_field]) in (rdflib.term.URIRef, rdflib.term.Literal):
                continue

            if triple.get(cand_field) is None:

                unresolved.append({
                    "triple_id": triple_idx,
                    "component": value_field,
                    "label": triple[value_field],
                })

    return unresolved

In [282]:
disambiguated_triples = copy.deepcopy(scored_triples)

apply_disambiguation(disambiguated_triples, disambiguated_components)
unresolved_components = collect_unresolved_components(disambiguated_triples)

In [283]:
unresolved_components

[{'triple_id': 0,
  'component': 'subject',
  'label': 'Decision No 30-53 of 2 May 1953'},
 {'triple_id': 1,
  'component': 'subject',
  'label': 'Decision No 30-53 of 2 May 1953'},
 {'triple_id': 1, 'component': 'predicate', 'label': 'concerns'},
 {'triple_id': 1,
  'component': 'object',
  'label': 'practices prohibited by Article 60 (1) of the Treaty'},
 {'triple_id': 2, 'component': 'subject', 'label': 'practices'},
 {'triple_id': 2,
  'component': 'object',
  'label': 'Article 60 (1) of the Treaty'},
 {'triple_id': 3,
  'component': 'subject',
  'label': 'Decision No 30-53 of 2 May 1953'},
 {'triple_id': 3, 'component': 'predicate', 'label': 'applies to'},
 {'triple_id': 3, 'component': 'object', 'label': 'common market'},
 {'triple_id': 4,
  'component': 'subject',
  'label': 'Decision No 30-53 of 2 May 1953'},
 {'triple_id': 4, 'component': 'predicate', 'label': 'applies to'},
 {'triple_id': 4, 'component': 'object', 'label': 'coal'},
 {'triple_id': 5,
  'component': 'subject',


### Triple Sanitizer (Under Review)

In [14]:
def safe_parse(triple):
    try:
        return json.loads(triple)
    except json.JSONDecodeError:
        return None

def repair_json(text: str):
    text = text.strip()

    # Remove trailing junk after last valid closing bracket
    match = re.search(r"(\{.*\}|\[.*\])", text)
    if match:
        text = match.group(0)

    # Fix common bracket imbalance
    open_brackets = text.count("[")
    close_brackets = text.count("]")
    if open_brackets > close_brackets:
        text += "]" * (open_brackets - close_brackets)

    open_braces = text.count("{")
    close_braces = text.count("}")
    if open_braces > close_braces:
        text += "}" * (open_braces - close_braces)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

In [15]:
cleaned_data = dict()

for filename, triple_list in triples_by_filename.items():
    parsed_triples = list()

    for triple in triple_list:
        triple = triple.strip()
        if not triple:
            continue

        parsed = safe_parse(triple)

        if parsed is None:
            parsed = repair_json(triple)

        if parsed is None:
            print("FAILED:", triple)
            continue

        parsed_triples.extend(parsed)

    cleaned_data[filename] = parsed_triples

len(cleaned_data)


99

##### *Inspecting the Cleaned Triples:*

In [16]:
total = 0

for triple_list in cleaned_data.values():
    total += len(triple_list)

total

4073

In [22]:
cleaned_data["31958D1127_01_en.md"]

[{'subject': {'name': 'cdm#document-ec',
   'labels': ['Rules of the Transport Committee']},
  'predicate': {'name': 'cdm#adopted_by', 'labels': []},
  'object': {'name': 'cdm#entity_council', 'labels': ['EEC Council']}},
 {'subject': {'name': 'cdm#document-ec', 'labels': []},
  'predicate': {'name': 'cdm#issued_by', 'labels': []},
  'object': {'name': 'cdm#official-journal', 'labels': []}},
 {'subject': {'name': 'cdm#document-ec', 'labels': []},
  'predicate': {'name': 'cdm#has_type', 'labels': []},
  'object': {'name': 'cdm#official-journal-act', 'labels': []}},
 {'subject': {'name': 'cdm#committee', 'labels': ['Transport Committee']},
  'predicate': {'name': 'cdm#has_part', 'labels': []},
  'object': {'name': 'cdm#person', 'labels': ['expert']}},
 {'subject': {'name': 'cdm#organization', 'labels': ['European Commission']},
  'predicate': {'name': 'cdm#consults', 'labels': []},
  'object': {'name': 'cdm#committee', 'labels': ['Transport Committee']}},
 {'subject': {'name': 'cdm#repor

In [73]:
subjs_objs = set()

for filename, triple_list in cleaned_data.items():
    for triple in triple_list:

        # clean subject
        subj = triple["subject"]["name"]
        subj_clean = strip_cdm_prefix(subj)
        triple["subject"]["name"] = subj_clean

        # clean object
        obj = triple["object"]["name"]
        obj_clean = strip_cdm_prefix(obj)
        triple["object"]["name"] = obj_clean

        # always use cleaned values
        subjs_objs.add(subj_clean)
        subjs_objs.add(obj_clean)

subjs_objs = list(subjs_objs)

### Entity-Relation Normalizer (INCOMPLETE)

In [284]:
def prepare_resolution_sets(unresolved_components):
    """
    Prepare the custom entity and relation pools for
    LSH + FAISS + LLM resolution.

    Parameters
    ----------
    unresolved_components : list[dict]
        Each element has the form:
        {
            "triple_id": int,
            "component": "subject" | "predicate" | "object",
            "label": str
        }

    Returns
    -------
    entity_labels : list[str]
        Unique unresolved entity labels.

    relation_labels : list[str]
        Unique unresolved relation labels.

    entity_refs : dict[str, list[(triple_id, component)]]
        Maps each entity label to all of its occurrences.

    relation_refs : dict[str, list[(triple_id, component)]]
        Maps each relation label to all of its occurrences.
    """

    entity_refs = defaultdict(list)
    relation_refs = defaultdict(list)

    for item in unresolved_components:

        label = item["label"].strip()
        triple_id = item["triple_id"]
        component = item["component"]

        if component == "predicate":
            relation_refs[label].append((triple_id, component))
        else:
            entity_refs[label].append((triple_id, component))

    entity_labels = sorted(entity_refs.keys())
    relation_labels = sorted(relation_refs.keys())

    return (
        entity_labels,
        relation_labels,
        entity_refs,
        relation_refs,
    )

In [285]:
(
    entity_labels,
    relation_labels,
    entity_refs,
    relation_refs,
) = prepare_resolution_sets(unresolved_components)

In [288]:
len(relation_labels)

1834

In [289]:
def ngram_tokenize(text, n=3):
    text = text.lower().replace(" ", "")
    return [
        text[i:i+n]
        for i in range(len(text) - n + 1)
    ]


class UnionFind:

    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra != rb:
            self.parent[rb] = ra


def build_similarity_clusters(
    labels,
    model,
    lsh_threshold=0.5,
    similarity_threshold=0.9,
    num_perm=256,
):
    """
    Returns:
        candidate_map
        final_pairs
        clusters
    """

    # -------------------------
    # LSH
    # -------------------------

    lsh = MinHashLSH(
        threshold=lsh_threshold,
        num_perm=num_perm,
    )

    minhash_store = {}

    for idx, label in enumerate(labels):

        m = MinHash(num_perm=num_perm)

        for token in ngram_tokenize(label):
            m.update(token.encode("utf8"))

        lsh.insert(idx, m)
        minhash_store[idx] = m

    candidate_map = {}

    for idx in range(len(labels)):

        candidate_map[idx] = [
            c
            for c in lsh.query(minhash_store[idx])
            if c != idx
        ]

    # -------------------------
    # FAISS
    # -------------------------

    embeddings = model.encode(
        labels,
        normalize_embeddings=True,
    ).astype(np.float32)

    final_pairs = set()

    for i, candidate_ids in candidate_map.items():

        if not candidate_ids:
            continue

        query_vec = embeddings[i].reshape(1, -1)

        candidate_vecs = embeddings[candidate_ids]

        scores = np.dot(
            candidate_vecs,
            query_vec.T,
        ).flatten()

        for j, score in zip(candidate_ids, scores):

            if score >= similarity_threshold and i < j:

                final_pairs.add(
                    (i, j, float(score))
                )

    # -------------------------
    # Union-Find clustering
    # -------------------------

    uf = UnionFind(len(labels))

    for i, j, _ in final_pairs:
        uf.union(i, j)

    grouped = defaultdict(list)

    for idx in range(len(labels)):
        grouped[uf.find(idx)].append(idx)

    clusters = [
        {
            "cluster_id": cluster_id,
            "labels": [labels[i] for i in members],
            "members": members,
        }
        for cluster_id, members in grouped.items()
    ]

    return candidate_map, final_pairs, clusters

In [293]:
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

(
    entity_candidate_map,
    entity_pairs,
    entity_clusters
    ) = build_similarity_clusters(
            entity_labels,
            emb_model,
            similarity_threshold=0.90,
            )


(
    relation_candidate_map,
    relation_pairs,
    relation_clusters
    ) = build_similarity_clusters(
            relation_labels,
            emb_model,
            similarity_threshold=0.95,
        )

2026-07-09 21:35:58,457 - INFO - Use pytorch device_name: cpu
2026-07-09 21:35:58,474 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches:   0%|          | 0/204 [00:00<?, ?it/s]

Batches:   0%|          | 0/58 [00:00<?, ?it/s]

In [292]:
entity_pairs

{(1835, 1837, 0.9189935922622681),
 (5549, 5550, 0.9813708066940308),
 (5511, 5981, 0.9288270473480225),
 (2243, 4492, 0.9067869782447815),
 (1471, 5303, 0.9060825109481812),
 (1263, 1264, 0.9453296661376953),
 (700, 2504, 0.9213166236877441),
 (434, 890, 0.9182927012443542),
 (888, 890, 0.9353832602500916),
 (399, 616, 0.9210686683654785),
 (213, 303, 0.9354166388511658),
 (512, 513, 0.9684674739837646),
 (5765, 6473, 0.9717431664466858),
 (1534, 1702, 0.9831185936927795),
 (5698, 5699, 0.9289703965187073),
 (559, 562, 0.9533128142356873),
 (4232, 4242, 0.9150188565254211),
 (695, 877, 0.986030101776123),
 (1020, 5843, 0.9655796885490417),
 (949, 2862, 1.000000238418579),
 (3142, 4433, 0.9492466449737549),
 (3354, 3355, 0.9230195879936218),
 (733, 4181, 0.9488709568977356),
 (1143, 1144, 1.0),
 (3813, 3815, 0.9990450739860535),
 (859, 861, 0.9793821573257446),
 (414, 418, 0.989173948764801),
 (2922, 2925, 0.9043000936508179),
 (608, 609, 0.9661795496940613),
 (875, 878, 0.955779790878

In [69]:
matched_list = []

for index, neighbor, score in final_pairs:
    if score == 1.0:
        matched_list.append((subjs_objs[index], subjs_objs[neighbor], score))

matched_list

[('court\u202fof\u202fjustice', 'Court of Justice', 1.0),
 ('directorsrepresentingelectricitedefrance',
  'DirectorsRepresentingElectricitéDeFrance',
  1.0),
 ('series\u202fi', 'series i', 1.0),
 ('director of directorate b of the directorate‑general for agriculture',
  'Director of Directorate\u202fB of the Directorate‑General for Agriculture',
  1.0)]

*Naive Deduplication:*

In [70]:
for triple_list in cleaned_data.values():
     sim_list = [
          (id, match[0].lower())
          for id, triple in enumerate(triple_list)
          for match in matched_list
          if (match[0].lower() == (triple["subject"]["name"].lower())) or (match[0].lower() == triple["object"]["name"].lower())
          ]

     for id, reference in sim_list:
          if triple_list[id]["subject"]["name"].lower() == reference:
               triple_list[id]["subject"]["name"] = reference
          if triple_list[id]["object"]["name"].lower() == reference:
               triple_list[id]["object"]["name"] = reference

          print(triple_list[id]["subject"]["name"], triple_list[id]["object"]["name"])

court of justice review body for penalties under Regulation No 11
danish_special_edition series i
english_special_edition series i
directorsrepresentingelectricitedefrance ElectriciteDeFrance
subparagraph (c) of Article 1 (2) director of directorate b of the directorate‑general for agriculture
director of directorate b of the directorate‑general for agriculture cdmplus#Agent


### Schema Validator (FUTURE WORK)

### Graph Serializer

In [100]:
KNOWN_PREDICATES = {
    "rdf:type": RDF.type,
    "type": RDF.type,
    "rdfs:label": RDFS.label,
    "label": RDFS.label,
    "owl:sameAs": OWL.sameAs
}

def normalize_uri(text):
    text = text.strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^\w\-\#]", "", text)  # keep #
    return text

def resolve_resource(p):
    # normalize
    key = p.strip()

    # ✅ case 1: known RDF predicate
    for known in KNOWN_PREDICATES.keys():
        if key.lower() == known.lower():
            key = known
            return KNOWN_PREDICATES[key]

    # ✅ case 2: in your ontology dict
    for resource in rdf_dict.keys():
        if key.lower() == resource.lower():
            key = resource
            return URIRef(rdf_dict[key])

    # ✅ case 3: fallback → your namespace
    return JS[key.replace(" ", "_")]

In [102]:
pure_triples = {filename: set() for filename in cleaned_data.keys()}
errors = {filename: list() for filename in cleaned_data.keys()}

JS = Namespace("http://jurisynth.org/cdmext/data/")

for filename, triple_list in copy.deepcopy(cleaned_data).items():
    for triple in triple_list:

        subj = normalize_uri(triple["subject"]["name"])
        obj = normalize_uri(triple["object"]["name"])

        pred_labels = triple["predicate"].get("labels", [])

        if pred_labels:
            for pred_tag in pred_labels:
                try:
                    pred = resolve_resource(pred_tag)

                    pure_triples[filename].add((JS[subj], pred, JS[obj]))

                except Exception:
                    errors[filename].append((JS[subj], pred_tag, JS[obj], 1))

        else:
            key = triple["predicate"]["name"]

            pred = resolve_resource(key)

            pure_triples[filename].add((JS[subj], pred, JS[obj]))

total = sum([len(triple_list) for triple_list in pure_triples.values()])
print("No. of proper triples:", total)

subj_tags = copy.deepcopy([
    (triple['subject']["name"], tuple(triple['subject']["labels"]), filename)
    for filename, triple_list in cleaned_data.items()
    for triple in triple_list
    ])
obj_tags = copy.deepcopy([
    (triple['object']["name"], tuple(triple['object']["labels"]), filename)
    for filename, triple_list in cleaned_data.items()
    for triple in triple_list
    ])
triple_tags = set(subj_tags + obj_tags)

empty_tags = set([pair for pair in triple_tags if len(pair[1]) == 0])
triple_tags -= empty_tags

for entity, label_list, filename in triple_tags:
    norm_entity = normalize_uri(entity)
    for label in label_list:
        clean_label = normalize_uri(strip_cdm_prefix(label))
        resolved_label = resolve_resource(clean_label)

        pure_triples[filename].add((
            JS[norm_entity],
            RDF.type,
            resolved_label
            ))

total = sum([len(triple_list) for triple_list in errors.values()])
print()
print("No. of erroneous triples:", total)

No. of proper triples: 4056

No. of erroneous triples: 0


*Issue (Future Work):*
* Route highly-matched resources to their appropriate namespaces.

In [103]:
pure_triples

{'31961D0408_01_en.md': {(rdflib.term.URIRef('http://jurisynth.org/cdmext/data/1961-03-08'),
   rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
   rdflib.term.URIRef('http://purl.org/dc/terms/date')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/addresses'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/compatibility_of_Italian_draft_law_with_common_market_under_Article_923c')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/date_adopted'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/1961-03-08')),
  (rdflib.term.URIRef('http://jurisynth.org/cdmext/data/Commission_Decision_on_modification_of_aid_system_in_Italy'),
   rdflib.term.URIRef('http://jurisynth.org/cdmext/data/issued_by'),
   rdfli

In [104]:
errors_list = list()

for err_list in errors.values():
    errors_list.extend(err_list)

len(errors_list)

0

*Handling Erroneous Triples (if any):*

In [140]:
def batch_list(lst, size=10):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def error_fix_batch(errors):
    batch_prompt = "Fix the labels for the following triples.\n\n"

    metadata = []

    for i, error in enumerate(errors):
        err_index = error[-1]
        incomplete_tag = error[err_index]
        
        if "#" not in incomplete_tag:
            continue  # or handle differently

        reference = incomplete_tag.split("#")[-1]
        triple = ", ".join(error[:3])
        query_list = [key for key in rdf_dict.keys() if reference in key]

        batch_prompt += f"""
        Item {i}:
        Triple: {triple}
        Incorrect label: {incomplete_tag}
        Candidates: {", ".join(query_list)}
        """

        metadata.append((i, error, err_index))

    batch_prompt += """
    Return the corrected labels as a JSON array of strings in order:
    ["label1", "label2", ...]
    Only return the JSON array.
    """

    # 🔑 ONE API CALL
    labels = json.loads(get_completion(system_prompt=batch_prompt, query=""))

    corrected = []

    for label, (_, error, err_index) in zip(labels, metadata):
        triple = list(error[:3])
        try:
            triple[err_index] = rdf_dict[label]
        except:
            print(triple, err_index)
        trp_dict = {"triple": triple, "filename": filename}
        corrected.append(triple)

    return corrected

In [ ]:
error_batches = list(batch_list(errors_list, 10))

with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(error_fix_batch, error_batches))

fixed_list = []
for batch in results:
    fixed_list.extend(batch)

In [105]:
named_graph = Dataset()
SOURCE = Namespace("http://jurisynth.org/cdmext/source/")

named_graph.bind("js", JS)
named_graph.bind("source", SOURCE)

for idx, ns in schema_namespaces:
    named_graph.bind(idx, ns)

for filename, triple_list in pure_triples.items():

    graph_uri = SOURCE[normalize_uri(filename)]
    g = named_graph.graph(graph_uri)

    for s, p, o in triple_list:
        g.add((s, p, o))
        
named_graph.serialize("eu_legislation_graph.nq", format="nquads")

<Graph identifier=N65eac2211c334231b44be1f3bfb8745d (<class 'rdflib.graph.Dataset'>)>